# Task 2: LLM-Powered Diagnostic Agent

**Loop:** belief state -> LLM decision -> action validation -> tool execution -> observation -> Bayesian update -> memory update -> next decision or stop

In [1]:
import math
import random
import json
import re

## 1. Problem Constants
Faults, tool costs, per-fault ABNORMAL probabilities, budget `C=5`, stopping confidence `0.90`

In [2]:
FAULTS = [
    "network_failure",
    "disk_full",
    "memory_exhaustion",
    "service_crash",
    "configuration_error",
]

# TOOLS[tool]["p_abnormal"][fault] = P(observation=ABNORMAL | fault, tool)
TOOLS = {
    "ping_server": {
        "cost": 1,
        "p_abnormal": {
            "network_failure": 0.95, "disk_full": 0.05, "memory_exhaustion": 0.05,
            "service_crash": 0.10, "configuration_error": 0.10,
        },
    },
    "check_disk": {
        "cost": 1,
        "p_abnormal": {
            "network_failure": 0.05, "disk_full": 0.95, "memory_exhaustion": 0.10,
            "service_crash": 0.10, "configuration_error": 0.10,
        },
    },
    "check_memory": {
        "cost": 1,
        "p_abnormal": {
            "network_failure": 0.05, "disk_full": 0.10, "memory_exhaustion": 0.95,
            "service_crash": 0.10, "configuration_error": 0.15,
        },
    },
    "read_service_log": {
        "cost": 2,
        "p_abnormal": {
            "network_failure": 0.10, "disk_full": 0.20, "memory_exhaustion": 0.25,
            "service_crash": 0.90, "configuration_error": 0.55,
        },
    },
    "check_config": {
        "cost": 2,
        "p_abnormal": {
            "network_failure": 0.05, "disk_full": 0.10, "memory_exhaustion": 0.10,
            "service_crash": 0.20, "configuration_error": 0.90,
        },
    },
}

BUDGET = 5
CONFIDENCE_THRESHOLD = 0.90

## 2. Environment
Secretly picks the hidden fault `H`; each tool call is a Bernoulli sample from the tool's probability table. The agent never reads `true_fault`.

In [3]:
class Environment:
    """Secretly picks one true fault H ~ Uniform(FAULTS); answers tool calls."""

    def __init__(self, seed):
        rng = random.Random(seed)
        self.true_fault = rng.choice(FAULTS)
        self._rng = rng

    def run_tool(self, tool):
        """TOOL: run_tool(a_t) -> o_t in {NORMAL, ABNORMAL}, Bernoulli-sampled."""
        p_ab = TOOLS[tool]["p_abnormal"][self.true_fault]
        return "ABNORMAL" if self._rng.random() < p_ab else "NORMAL"

## 3. Belief state, Bayes update, entropy, information gain
The mathematical core: `bayes_update`, `entropy`, `info_gain` 

In [4]:
def uniform_belief():
    return {h: 1.0 / len(FAULTS) for h in FAULTS}


def likelihood(tool, obs, fault):
    p_ab = TOOLS[tool]["p_abnormal"][fault]
    return p_ab if obs == "ABNORMAL" else (1.0 - p_ab)


def bayes_update(belief, tool, obs):
    """
    P(H=h | o_t,a_t,M_{t-1}) = P(o_t|H=h,a_t) P(H=h|M_{t-1}) / Z
    -- exactly the formula in the assignment, applied via plain dict ops.
    """
    unnorm = {h: likelihood(tool, obs, h) * belief[h] for h in FAULTS}
    z = sum(unnorm.values())
    if z < 1e-15:
        # Rather than silently divide by ~0 -- a real failure mode worth naming.
        return uniform_belief()
    return {h: v / z for h, v in unnorm.items()}


def entropy(belief):
    """H(b_t) = -sum_h P(h) log2 P(h), with the 0*log(0)=0 convention."""
    return -sum(p * math.log2(p) for p in belief.values() if p > 1e-15)


def predictive_obs_prob(belief, tool):
    """P(o | a, M_t) marginalized over the current belief over h."""
    p_ab = sum(TOOLS[tool]["p_abnormal"][h] * belief[h] for h in FAULTS)
    return {"ABNORMAL": p_ab, "NORMAL": 1.0 - p_ab}


def info_gain(belief, tool):
    """IG(a) = H(b_t) - sum_o P(o|a,M_t) H(b_t+1^(o))."""
    h_now = entropy(belief)
    preds = predictive_obs_prob(belief, tool)
    expected_h_next = 0.0
    for obs, p_o in preds.items():
        if p_o <= 1e-15:
            continue
        b_next = bayes_update(belief, tool, obs)
        expected_h_next += p_o * entropy(b_next)
    return h_now - expected_h_next

## 4. Episode state
Memory `M_t`, belief `b_t`, used tools, budget and counters (LLM calls, invalid proposals).

In [5]:
class EpisodeState:
    def __init__(self, budget=BUDGET):
        self.memory = []            # [(a_1,o_1), (a_2,o_2), ...]
        self.belief = uniform_belief()
        self.used_tools = set()
        self.budget_remaining = budget
        self.total_cost = 0
        self.n_llm_calls = 0
        self.n_invalid = 0

    def affordable_unused_tools(self):
        return [t for t in TOOLS
                if t not in self.used_tools and TOOLS[t]["cost"] <= self.budget_remaining]

    def apply_action(self, tool, obs):
        self.memory.append((tool, obs))
        self.belief = bayes_update(self.belief, tool, obs)
        self.used_tools.add(tool)
        cost = TOOLS[tool]["cost"]
        self.budget_remaining -= cost
        self.total_cost += cost

## 5. Policies
Random baseline and greedy `IG(a)/c(a)` baseline.

In [6]:
def random_policy(state, rng):
    options = state.affordable_unused_tools()
    return rng.choice(options) if options else "STOP"


def greedy_ig_policy(state):
    """Baseline: argmax_a IG(a)/cost(a) over affordable, unused tools."""
    options = state.affordable_unused_tools()
    if not options:
        return "STOP"
    scores = {t: info_gain(state.belief, t) / TOOLS[t]["cost"] for t in options}
    return max(scores, key=scores.get)

## 6. LLM policy with Closed-Loop Controller Feedback
Prompt contains only legitimately available info. `llm_policy_step` implements an **accumulative reflective retry loop**: when the controller rejects an action, it logs and accumulates the rejection diagnostics across retry attempts, preventing the LLM from repeating mistakes or oscillating between invalid choices within the same decision step.

In [7]:
# --- LLM policy: prompt construction, API call, response parsing ---------

def build_llm_prompt(state, rejection_feedback=None):
    """
    Only information legitimately available to the agent, per the
    assignment: current posteriors, history, unused tools + costs,
    remaining budget, permissible actions. No peeking at the true fault.
    Includes controller rejection feedback on retries to allow self-correction.
    """
    belief_str = ", ".join(f"{h}: {p:.3f}" for h, p in state.belief.items())
    history_str = "; ".join(f"{a}->{o}" for a, o in state.memory) or "none yet"
    options = state.affordable_unused_tools()
    options_str = ", ".join(f"{t} (cost {TOOLS[t]['cost']})" for t in options) or "none"

    feedback_section = ""
    if rejection_feedback:
        feedback_section = f"""
[CONTROLLER REJECTION FEEDBACK]
{rejection_feedback}
"""

    return f"""You are the decision-making policy of a server-diagnostic agent.
Choose exactly one action: the name of one unused, affordable diagnostic tool,
or the literal string "STOP".

Current posterior beliefs over the hidden fault:
{belief_str}

Diagnostic history so far (action -> observation):
{history_str}

Remaining budget: {state.budget_remaining}
Affordable unused tools: {options_str}
{feedback_section}
Rules:
- You may only choose a tool from the affordable unused tools listed above, or "STOP".
- You may ONLY propose "STOP" if maximum posterior belief >= 0.90, or if no affordable unused tools remain. Otherwise, you MUST select an affordable unused tool.
- Respond with ONLY a JSON object, no extra text, in exactly this format:
  {{"action": "<tool_name_or_STOP>", "reason": "<one short sentence>"}}
"""


def call_qwen(prompt, client, temperature=0.0):
    """Calls Qwen instruction model via client with configurable temperature."""
    response = client.chat_completion(
        messages=[{"role": "user", "content": prompt}],
        max_tokens=120,
        temperature=temperature,
    )
    return response.choices[0].message.content


def parse_llm_response(raw_text):
    """
    Robust JSON parser:
    1. Strips reasoning/thinking blocks (<think>...</think>) if present.
    2. Bypasses markdown code fences (```json ... ```) and surrounding chatter.
    3. Validates strictly formatted JSON with an 'action' key.
    """
    if not raw_text:
        return None
    # Strip <think>...</think> blocks
    cleaned = re.sub(r"<think>.*?</think>", "", raw_text, flags=re.DOTALL).strip()
    # Target JSON containing 'action' key
    match = re.search(r'\{[^{}]*"action"[^{}]*\}', cleaned, re.DOTALL)
    if not match:
        match = re.search(r"\{.*\}", cleaned, re.DOTALL)
    if not match:
        return None
    try:
        parsed = json.loads(match.group(0))
    except json.JSONDecodeError:
        return None
    if "action" not in parsed or not isinstance(parsed["action"], str):
        return None
    return parsed


def llm_policy_step(state, client, max_retries=3):
    """
    Calls the LLM, validates its proposal, and retries on rejection with
    accumulated controller feedback -- maintaining a history of rejected attempts
    within this decision step so the LLM does not repeat mistakes across retries.
    """
    feedbacks = []
    options = state.affordable_unused_tools()
    options_str = ", ".join(f"{t} (cost {TOOLS[t]['cost']})" for t in options) or "none"

    for attempt in range(max_retries):
        state.n_llm_calls += 1
        feedback_text = "\n".join(feedbacks) if feedbacks else None
        prompt = build_llm_prompt(state, rejection_feedback=feedback_text)
        temp = 0.0 if attempt == 0 else 0.2
        raw = call_qwen(prompt, client, temperature=temp)
        parsed = parse_llm_response(raw)
        proposed = parsed["action"] if parsed else "MALFORMED"

        valid, reason = validate_action(proposed, state)
        if valid:
            return proposed
        state.n_invalid += 1

        # Generate informative controller feedback for the next attempt
        if reason == "premature_stop":
            max_conf = max(state.belief.values())
            msg = f"- Attempt {attempt + 1}: Action 'STOP' rejected: Confidence is only {max_conf:.2f} (< 0.90) and affordable tools remain ({options_str}). You must pick an affordable tool."
        elif reason == "over_budget":
            cost = TOOLS.get(proposed, {}).get("cost", "?")
            msg = f"- Attempt {attempt + 1}: Action '{proposed}' rejected: Cost ({cost}) exceeds remaining budget ({state.budget_remaining}). Choose from affordable tools: {options_str}."
        elif reason == "repeated_tool":
            msg = f"- Attempt {attempt + 1}: Action '{proposed}' rejected: Tool already used. Each tool may be used at most once. Choose from unused tools: {options_str}."
        elif reason == "unknown_tool":
            msg = f"- Attempt {attempt + 1}: Action '{proposed}' rejected: Unknown tool. Valid options are: {options_str} or STOP."
        else: # malformed
            msg = f"- Attempt {attempt + 1}: Action rejected: Response was not valid JSON matching {{'action': '...', 'reason': '...'}}. Output strictly JSON."
        feedbacks.append(msg)

    return greedy_ig_policy(state)


## 7. Controller and Stopping Logic
### • `should_stop(state)`: Unified stopping condition per spec (terminates when $\max_h P(H=h) \ge 0.90$ or budget exhausted).
### • `validate_action(proposed, state)`: Validation gate rejecting malformed output, unknown/repeated tools, over-budget actions, and premature `STOP`.

In [8]:
def should_stop(state):
    """
    Unified stopping condition per specification:
    Terminates when max_h P(H=h | M_t) >= 0.90, or when no affordable unused tool remains.
    Returns (True, reason) if stopping condition is met, else (False, "continue").
    """
    max_conf = max(state.belief.values())
    if max_conf >= CONFIDENCE_THRESHOLD:
        return True, "confident_stop"
    if not state.affordable_unused_tools():
        return True, "no_tools_remaining"
    return False, "continue"


def validate_action(proposed, state):
    """
    Deterministic gate required by spec: rejects malformed output,
    unavailable tools, repeated tool calls, over-budget actions, and
    premature STOP while confidence < 0.90 and an affordable tool remains.
    """
    if proposed == "STOP":
        met, reason = should_stop(state)
        return (True, reason) if met else (False, "premature_stop")

    if proposed not in TOOLS:
        return False, "unknown_tool"
    if proposed in state.used_tools:
        return False, "repeated_tool"
    if TOOLS[proposed]["cost"] > state.budget_remaining:
        return False, "over_budget"

    return True, "ok"


## 8. Episode runner
One full episode: policy -> validate -> execute -> Bayes update -> memory update -> stop check. Also computes `J = 100*1{wrong} + cost + 0.1*N_LLM`.

In [9]:
def run_episode(policy_name, seed, llm_client=None, budget=BUDGET):
    env = Environment(seed)
    state = EpisodeState(budget=budget)
    rng = random.Random(seed + 1_000_000)  # separate stream from env's own rng

    while True:
        if policy_name == "random":
            action = random_policy(state, rng)
        elif policy_name == "greedy":
            action = greedy_ig_policy(state)
        elif policy_name == "llm":
            action = llm_policy_step(state, llm_client)
        else:
            raise ValueError(f"unknown policy {policy_name}")

        if action == "STOP":
            break

        obs = env.run_tool(action)
        state.apply_action(action, obs)

        # Single source of truth for stopping
        stop_now, _ = should_stop(state)
        if stop_now:
            break

    h_hat = max(state.belief, key=state.belief.get)
    correct = (h_hat == env.true_fault)
    J = 100 * (0 if correct else 1) + state.total_cost + 0.1 * state.n_llm_calls

    return {
        "correct": correct,
        "true_fault": env.true_fault,
        "predicted_fault": h_hat,
        "total_cost": state.total_cost,
        "n_tool_calls": len(state.memory),
        "final_confidence": max(state.belief.values()),
        "n_llm_calls": state.n_llm_calls,
        "n_invalid": state.n_invalid,
        "J": J,
    }


## 9. Evaluation harness
Seeded episodes per policy; reports accuracy, avg cost, avg tool calls, final confidence, LLM calls, invalid actions, avg `J`.

In [10]:
def evaluate_policy(policy_name, n_episodes=200, seed_start=0, llm_client=None, checkpoint_file=None):
    results = []
    start_idx = 0
    if checkpoint_file and os.path.exists(checkpoint_file):
        try:
            with open(checkpoint_file, "r") as f:
                results = json.load(f)
            start_idx = len(results)
            print(f"  Resuming {policy_name} from checkpoint: {start_idx}/{n_episodes} completed.")
        except Exception:
            results = []
            start_idx = 0

    for i in range(start_idx, n_episodes):
        r = run_episode(policy_name, seed_start + i, llm_client=llm_client)
        results.append(r)
        if checkpoint_file and policy_name == "llm":
            with open(checkpoint_file, "w") as f:
                json.dump(results, f)
        if policy_name == "llm" and ((i + 1) % 10 == 0 or (i + 1) == n_episodes):
            print(f"  [LLM Progress] {i + 1}/{n_episodes} episodes completed...")

    n = len(results)
    return {
        "policy": policy_name,
        "n_episodes": n,
        "accuracy": sum(r["correct"] for r in results) / n,
        "avg_cost": sum(r["total_cost"] for r in results) / n,
        "avg_tool_calls": sum(r["n_tool_calls"] for r in results) / n,
        "avg_final_confidence": sum(r["final_confidence"] for r in results) / n,
        "avg_llm_calls": sum(r["n_llm_calls"] for r in results) / n,
        "avg_invalid": sum(r["n_invalid"] for r in results) / n,
        "avg_J": sum(r["J"] for r in results) / n,
    }


def print_summary(summary):
    print(f"\n=== Policy: {summary['policy']} ({summary['n_episodes']} episodes) ===")
    print(f"  accuracy             : {summary['accuracy']:.3f}")
    print(f"  avg cost             : {summary['avg_cost']:.3f}")
    print(f"  avg tool calls       : {summary['avg_tool_calls']:.3f}")
    print(f"  avg final confidence : {summary['avg_final_confidence']:.3f}")
    print(f"  avg LLM calls        : {summary['avg_llm_calls']:.3f}")
    print(f"  avg invalid actions  : {summary['avg_invalid']:.3f}")
    print(f"  avg J (loss)         : {summary['avg_J']:.3f}")

## 10. Offline baselines (Random vs. Greedy Information Gain)
Evaluated over 200 seeded episodes ($i = 0 \dots 199$) for strict apples-to-apples comparison with the LLM policy.

In [11]:
summary_random = evaluate_policy("random", n_episodes=200, seed_start=0)
summary_greedy = evaluate_policy("greedy", n_episodes=200, seed_start=0)
print_summary(summary_random)
print_summary(summary_greedy)



=== Policy: random (200 episodes) ===
  accuracy             : 0.740
  avg cost             : 4.980
  avg tool calls       : 3.655
  avg final confidence : 0.711
  avg LLM calls        : 0.000
  avg invalid actions  : 0.000
  avg J (loss)         : 30.980

=== Policy: greedy (200 episodes) ===
  accuracy             : 0.765
  avg cost             : 5.000
  avg tool calls       : 4.000
  avg final confidence : 0.780
  avg LLM calls        : 0.000
  avg invalid actions  : 0.000
  avg J (loss)         : 28.500


## 11. Test suite: mathematical verification and controller self-test (offline)
1. **Mathematical unit tests** verifying Bayes posterior updates, entropy, and information gain against analytical ground truth.
2. **Controller failure simulator** (`ScriptedClient`) proving rejection of malformed JSON, premature `STOP`, unknown tools, and repeated tool calls.

In [12]:
# --- 1. Mathematical Unit Tests ------------------------------------------

def test_bayes_math():
    b0 = uniform_belief()
    # P(ABNORMAL | fault, ping_server) = [0.95, 0.05, 0.05, 0.10, 0.10]
    # Sum = 1.25 -> P(network_failure | ABNORMAL) = 0.95 / 1.25 = 0.760
    b1 = bayes_update(b0, "ping_server", "ABNORMAL")
    assert abs(b1["network_failure"] - 0.76) < 1e-4, f"Expected 0.76, got {b1['network_failure']}"
    assert abs(sum(b1.values()) - 1.0) < 1e-6, "Posterior must sum to 1"
    print("✓ test_bayes_math: Passed (network posterior surged to 76.0%).")

def test_entropy_and_ig():
    b0 = uniform_belief()
    h0 = entropy(b0)
    assert abs(h0 - math.log2(5)) < 1e-5, f"Entropy of 5 uniform faults must be log2(5)=2.3219, got {h0}"
    ig = info_gain(b0, "ping_server")
    assert ig > 0.0, f"Expected positive info gain, got {ig}"
    print(f"✓ test_entropy_and_ig: Passed (H={h0:.3f} bits, ping_server IG={ig:.3f} bits).")

def test_controller_invariants():
    state = EpisodeState(budget=1)
    val, reason = validate_action("read_service_log", state)
    assert not val and reason == "over_budget", f"Should reject over budget: {reason}"
    val, reason = validate_action("STOP", state)
    assert not val and reason == "premature_stop", f"Should reject premature STOP: {reason}"
    print("✓ test_controller_invariants: Passed (budget and premature STOP enforced).")

test_bayes_math()
test_entropy_and_ig()
test_controller_invariants()

# --- 2. Controller Self-Test with Scripted Fake LLM ----------------------

class FakeResponse:
    def __init__(self, text):
        self.choices = [type("C", (), {"message": type("M", (), {"content": text})()})()]

class ScriptedClient:
    """Mimics InferenceClient.chat_completion with a fixed reply script."""
    def __init__(self, replies):
        self.replies = list(replies)
        self.calls = 0
    def chat_completion(self, **kwargs):
        self.calls += 1
        text = self.replies.pop(0) if self.replies else '{"action": "STOP", "reason": "script exhausted"}'
        return FakeResponse(text)

scripted = ScriptedClient([
    "I think we should ping the server",                              # malformed (no JSON)
    '{"action": "STOP", "reason": "done"}',                          # premature STOP
    '{"action": "check_cpu", "reason": "?"}',                        # unknown tool
    '{"action": "ping_server", "reason": "check network first"}',    # valid
    '{"action": "ping_server", "reason": "again"}',                  # repeated tool
    '{"action": "check_disk", "reason": "next cheapest"}',           # valid
])
res = run_episode("llm", seed=0, llm_client=scripted)
print(res)
assert res["n_invalid"] >= 4, "controller should have rejected the 4 bad proposals"
print("\nController self-test passed: invalid proposals =", res["n_invalid"], "| LLM calls =", res["n_llm_calls"])


✓ test_bayes_math: Passed (network posterior surged to 76.0%).
✓ test_entropy_and_ig: Passed (H=2.322 bits, ping_server IG=0.452 bits).
✓ test_controller_invariants: Passed (budget and premature STOP enforced).
{'correct': True, 'true_fault': 'service_crash', 'predicted_fault': 'service_crash', 'total_cost': 5, 'n_tool_calls': 4, 'final_confidence': 0.755568871003004, 'n_llm_calls': 12, 'n_invalid': 11, 'J': 6.2}

Controller self-test passed: invalid proposals = 11 | LLM calls = 12


## 12. Real LLM policy (Qwen via Groq API)

> **Model Selection & Specification Compliance**:
> The assignment recommends `Qwen/Qwen3-4B-Instruct-2507`. Because Hugging Face has migrated free serverless endpoints to paid third-party provider tiers (`router.huggingface.co` returning HTTP 402), we access the open Qwen instruction model family via Groq's API (`qwen/qwen3.8-27b`). While having 27B parameters rather than 4B, it belongs to the same open-weights Qwen series, adheres to the exact same prompt structure and zero-framework constraints, and is evaluated here over **200 episodes** per the assignment specification.

In [ ]:
import os, time, urllib.request, json, re

class GroqClient:
    """Zero-dependency client mimicking chat_completion using Groq API with proactive pacing and robust rate-limit/timeout handling."""
    def __init__(self, api_key=None, model="qwen/qwen3.8-27b"):
        self.api_key = api_key or os.environ.get("GROQ_API_KEY")
        if not self.api_key:
            raise ValueError(
                "GROQ_API_KEY not found in environment!\n"
                "Please run `export GROQ_API_KEY='your_key'` in your terminal before running this cell."
            )
        self.model = model

    def chat_completion(self, messages, model=None, max_tokens=120, temperature=0.0):
        actual_model = self.model if (not model or "Qwen" in model) else model
        url = "https://api.groq.com/openai/v1/chat/completions"
        payload = {
            "model": actual_model,
            "messages": messages,
            "temperature": temperature,
            "max_tokens": max_tokens,
        }
        for attempt in range(25):
            try:
                time.sleep(1.2)  # Pacing to adhere strictly to 8,000 TPM limit
                req = urllib.request.Request(
                    url,
                    data=json.dumps(payload).encode("utf-8"),
                    headers={
                        "Authorization": f"Bearer {self.api_key}",
                        "Content-Type": "application/json",
                        "User-Agent": "Mozilla/5.0"
                    }
                )
                with urllib.request.urlopen(req, timeout=35) as resp:
                    data = json.loads(resp.read().decode())
                    text = data["choices"][0]["message"]["content"]
                    return type("Resp", (), {"choices": [type("C", (), {"message": type("M", (), {"content": text})()})()]})()
            except (TimeoutError, urllib.error.URLError, ConnectionError, OSError) as e:
                if attempt < 24:
                    wait_time = 3.0 * (attempt + 1)
                    time.sleep(wait_time)
                    continue
                raise
            except urllib.error.HTTPError as e:
                if e.code == 429 and attempt < 24:
                    wait_time = 22.0
                    if "retry-after" in e.headers:
                        try:
                            wait_time = max(float(e.headers["retry-after"]) + 1.0, wait_time)
                        except ValueError:
                            pass
                    else:
                        try:
                            body = e.read().decode()
                            err_body = json.loads(body)
                            m = re.search(r"try again in ([\d\.]+)s", err_body.get("error", {}).get("message", ""))
                            if m:
                                wait_time = max(float(m.group(1)) + 1.0, wait_time)
                        except Exception:
                            pass
                    time.sleep(wait_time)
                    continue
                raise

client = GroqClient()
# Evaluate LLM agent over 200 episodes with checkpointing
print("Evaluating Qwen on Groq (200 episodes)...")
checkpoint_path = os.path.join(os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() else ".", "llm_checkpoint_200.json")
summary_llm = evaluate_policy("llm", n_episodes=200, seed_start=0, llm_client=client, checkpoint_file=checkpoint_path)
print_summary(summary_llm)

Evaluating Qwen on Groq (200 episodes)...
  Resuming llm from checkpoint: 189/200 completed.
  [LLM Progress] 190/200 episodes completed...
  [LLM Progress] 200/200 episodes completed...

=== Policy: llm (200 episodes) ===
  accuracy             : 0.755
  avg cost             : 5.000
  avg tool calls       : 3.750
  avg final confidence : 0.789
  avg LLM calls        : 3.750
  avg invalid actions  : 0.000
  avg J (loss)         : 29.875


## 13. Comparison table
Run after the LLM cell(s) above.

In [14]:
rows = [summary_random, summary_greedy]
if 'summary_llm' in globals():
    rows.append(summary_llm)

print(f"{'policy':<20} {'acc':>6} {'cost':>6} {'calls':>6} {'conf':>6} {'llm':>6} {'invalid':>8} {'J':>8}")
print("-" * 70)
for s in rows:
    p_name = s['policy'][:19]
    print(f"{p_name:<20} {s['accuracy']:>6.3f} {s['avg_cost']:>6.2f} {s['avg_tool_calls']:>6.2f} "
          f"{s['avg_final_confidence']:>6.3f} {s['avg_llm_calls']:>6.2f} {s['avg_invalid']:>8.2f} {s['avg_J']:>8.2f}")


policy                  acc   cost  calls   conf    llm  invalid        J
----------------------------------------------------------------------
random                0.740   4.98   3.65  0.711   0.00     0.00    30.98
greedy                0.765   5.00   4.00  0.780   0.00     0.00    28.50
llm                   0.755   5.00   3.75  0.789   3.75     0.00    29.88
